# Ball detection: filtered RF-DETR + Kalman on both clips
Standalone notebook: reads `MyDrive/TacticalAI/data/raw/attack_clip.mp4` and
`attack_clip2.mp4`, runs the selected variants, tracks the ball, and **plays every result inline**.

| # | Model | What it is | Needs |
|---|---|---|---|
| 1 | **WASB-SBDT** (BMVC 2023) | Sports-ball heatmap network, sees 3 consecutive frames; official **basketball** weights (NBA broadcast) | nothing |
| 2 | **RF-DETR basketball** (Roboflow `basketball-player-detection-3-ycjdo/13`) | Requested RF-DETR medium checkpoint; architecture checked at runtime; full frame and 2x2 tiles | secret `ROBOFLOW_API_KEY` |
| 3 | **SAM 3** video, prompt `"basketball"` | Meta's text-prompted detector + video tracker | secret `HF_TOKEN`, after accepting terms at huggingface.co/facebook/sam3 |

**How to run in VS Code:** select your **Colab GPU kernel**, then **Run All**.
In browser Colab: Runtime -> Change runtime type -> **GPU**, then **Run all**.
For the default RF-DETR experiment, add your **private Roboflow API key** to
**Colab Secrets** as `ROBOFLOW_API_KEY` and enable notebook access. Get it at
https://app.roboflow.com/settings/api . Do not paste it into notebook source or chat.
If Secrets are unavailable in VS Code, setup asks for the key in a hidden prompt;
that value is kept only in the current runtime session.
RF-DETR uses local Colab inference; model authorization/download requires the key.
Re-run from setup so the new variant definitions and output directory are used.

**Current experiment:** `RF-DETR-tiles` baseline versus **Kalman + motion gates + short-gap interpolation**
on **both clips**, using saved detections from each clip. Only Kalman-filtered videos
are displayed by default; baseline metrics/videos are still saved.
This is the second individual video from the previous sweep, selected from visual
feedback, not established as the most accurate model. Settings are unchanged:
checkpoint `/13`, full frame + four overlapping 60% crops, detector cutoff 0.10,
legacy tracker cutoff 0.25, six-frame gap, and 0.6 reinitialization confidence.
The Kalman comparison requires three coherent frames to start, rejects out-of-gate
measurements regardless of confidence, predicts for at most 0.20 s, and interpolates
only gaps up to 0.16 s between accepted measurements in the same segment.
Its gates use image pixels, not physical court speed. They can reject real shots,
bounces, or camera motion; missing detections and incorrect initial locks remain possible.
`RF_REUSE_DIRS` points to each saved run; missing caches trigger inference only for
the missing clip. Set it to `{}` for fresh inference on both clips.
Cached detection time is historical, not time spent in this CPU postprocessing run.
WASB, SAM 3, and the other RF-DETR variants are skipped. No manual correction or
checkpoint location enters inference. A new output folder preserves previous runs.
To restore the clip-1 sweep, set `RF_CLIP_NAMES = ('attack_clip',)`,
`RF_SELECTED_VARIANTS = None`, and `RF_SWEEP = True`.

Set `EXPERIMENT = 'lowres'` to repeat the five low-resolution WASB variants,
using the previous final
`WASB-tiles-020` configuration as the reference. This reference choice assumes
"last test" meant that variant; visual feedback is not verified accuracy.
Set `EXPERIMENT = 'original'` in setup to restore the previous six-variant sweep
on both clips, including the unchanged clip 2 configuration. New run directories
preserve previous Drive artifacts. SAM 3 is skipped for this sweep (`RUN_OPTIONAL_MODELS = False`). Enable it separately
after model access is approved. RF-DETR remains enabled for the default experiment.
Each variant produces a full-length video, JSON evidence, and summary row.
The final cells show synchronized comparisons of the models run in this session.

**In the videos:** raw/rejected candidate rings are hidden (`SHOW_RAW_CANDIDATES = False`);
green ring = accepted measurement / filtered position; cyan ring labelled inferred =
offline interpolation; gray X = bounded Kalman extrapolation. Inferred points have
null measurement/confidence and are never counted as observed hits. Green trails
break at gaps and new segments. No interpolation across resets or long occlusions. Accepted false detections can
still appear; filtering is not verified ground truth. Raw/rejected evidence stays in JSON.
Outputs (videos, detections JSON, summary) are saved to `MyDrive/TacticalAI/outputs/ball-notebook-<time>/`.

In [ ]:
# Setup (installs first, before anything is imported)
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'transformers', 'accelerate',
                'omegaconf', 'gdown', 'uv'], check=True)

import json, math, os, time
import hashlib, importlib.metadata, platform
from datetime import datetime
from pathlib import Path
import cv2
import numpy as np
import torch
from IPython.display import Video, display, Markdown
from google.colab import drive, userdata

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TacticalAI')
EXPERIMENT = 'rfdet'  # 'rfdet': RF-DETR (optional WASB reference); 'lowres': WASB tweaks; 'original': both clips
RF_MODEL_ID = 'basketball-player-detection-3-ycjdo/13'
RF_SWEEP = False  # Used when RF_SELECTED_VARIANTS is None.
RF_SELECTED_VARIANTS = ('RF-DETR-tiles',)  # Second individual video from the prior sweep.
RF_CLIP_NAMES = ('attack_clip', 'attack_clip2')
RF_REUSE_DIRS = {  # Set to {} to force fresh inference on both clips.
    'attack_clip': ROOT / 'outputs' / 'ball-notebook-20261026-205848',
    'attack_clip2': ROOT / 'outputs' / 'ball-notebook-20261026-210537',
}
SHOW_RAW_CANDIDATES = False  # Rejected/raw candidates remain in JSON evidence.
SHOW_BASELINE = False  # Compute baseline metrics, display only Kalman videos.
COMPARE_KALMAN = True
KALMAN_SETTINGS = dict(confirmation_frames=3, max_gap_s=0.20, interpolation_gap_s=0.16,
                       max_speed_diagonal_s=1.0, max_innovation_diagonal=0.04,
                       measurement_std_diagonal=0.005, acceleration_std_diagonal_s2=2.0,
                       mahalanobis_squared_gate=9.21)
assert EXPERIMENT in ('rfdet', 'lowres', 'original')
CLIP_NAMES = (RF_CLIP_NAMES if EXPERIMENT == 'rfdet' else
              ('attack_clip', 'attack_clip2') if EXPERIMENT == 'original' else ('attack_clip',))
CLIPS = {name: ROOT / 'data/raw' / f'{name}.mp4' for name in CLIP_NAMES}
for path in CLIPS.values():
    assert path.is_file(), f'Missing video: {path}'
MODELS_DIR = ROOT / 'models'
MODELS_DIR.mkdir(parents=True, exist_ok=True)
OUT = ROOT / 'outputs' / ('ball-notebook-' + datetime.now().strftime('%Y%m%d-%H%M%S'))
OUT.mkdir(parents=True)

RF_CACHE_SOURCES = {clip: RF_REUSE_DIRS[clip] / f'rf_raw_{clip}.json'
                    for clip in CLIPS if clip in RF_REUSE_DIRS
                    and (RF_REUSE_DIRS[clip] / f'rf_raw_{clip}.json').is_file()}
RF_MISSING_CLIPS = {clip: path for clip, path in CLIPS.items() if clip not in RF_CACHE_SOURCES}
RF_CACHE_AVAILABLE = not RF_MISSING_CLIPS
assert RF_CACHE_AVAILABLE or torch.cuda.is_available(), 'Select a GPU runtime for fresh inference'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, '| torch', torch.__version__)
print('Output folder:', OUT)

def secret(name):
    try:
        return userdata.get(name)
    except Exception:
        return None

ROBOFLOW_API_KEY, HF_TOKEN = secret('ROBOFLOW_API_KEY'), secret('HF_TOKEN')
print('ROBOFLOW_API_KEY:', 'found' if ROBOFLOW_API_KEY else 'not loaded (not needed for cached postprocessing)')
print('HF_TOKEN:', 'found' if HF_TOKEN else 'not loaded (only needed if SAM 3 is enabled)')
if EXPERIMENT == 'rfdet' and not ROBOFLOW_API_KEY and not RF_CACHE_AVAILABLE:
    from getpass import getpass
    ROBOFLOW_API_KEY = getpass('Private Roboflow API key (hidden, session only): ').strip()
    if not ROBOFLOW_API_KEY:
        raise RuntimeError('A private Roboflow key is needed. Add it in Colab Secrets or the hidden setup prompt.')

# Minimum confidence for the tracker; WASB variant values are set in its experiment cell.
MIN_CONF = {'WASB': 0.5, 'RF-DETR': 0.25, 'SAM3': 0.3}
RUN_WASB = False  # Already checked; skip downloads, setup and inference.
RUN_OPTIONAL_MODELS = False  # Focus this run on RF-DETR; SAM 3 access was pending.
RUN_CONFIGS = {}
versions = {}
for package in ('torch', 'numpy', 'opencv-python', 'omegaconf', 'transformers', 'gdown'):
    try:
        versions[package] = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        versions[package] = None
(OUT / 'environment.json').write_text(json.dumps(dict(
    python=platform.python_version(), packages=versions, cuda=torch.version.cuda,
    gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None, offline=True, reid=False, gmc=False,
    experiment=EXPERIMENT, clips=CLIP_NAMES, run_wasb=RUN_WASB,
    run_optional_models=RUN_OPTIONAL_MODELS, rf_sweep=RF_SWEEP,
    rf_selected_variants=RF_SELECTED_VARIANTS, compare_kalman=COMPARE_KALMAN,
    kalman_settings=KALMAN_SETTINGS,
    rf_cache_sources={clip: str(path) for clip, path in RF_CACHE_SOURCES.items()},
    show_raw_candidates=SHOW_RAW_CANDIDATES, show_baseline=SHOW_BASELINE,
    assumptions=['one ball', 'constant-velocity association'],
    evaluation='19 draft development checkpoints; not held-out accuracy'), indent=2))

In [ ]:
# Shared helpers: video IO, ball trackers, drawing, scoring.
"""CPU-only image-space Kalman association and bounded offline gap interpolation.

No court calibration or evaluation labels enter this module. Motion limits are
configurable image-space heuristics, not physical basketball speed measurements.
"""
from dataclasses import asdict, dataclass
import math

import numpy as np


@dataclass(frozen=True)
class TemporalConfig:
    min_confidence: float = 0.25
    confirmation_frames: int = 3
    max_gap_s: float = 0.20
    interpolation_gap_s: float = 0.16
    max_speed_diagonal_s: float = 1.0
    max_innovation_diagonal: float = 0.04
    measurement_std_diagonal: float = 0.005
    acceleration_std_diagonal_s2: float = 2.0
    mahalanobis_squared_gate: float = 9.21

    def __post_init__(self):
        if any(not math.isfinite(v) or v <= 0 for v in asdict(self).values()):
            raise ValueError('Temporal parameters must be positive and finite')
        if not isinstance(self.confirmation_frames, int) or self.confirmation_frames < 2:
            raise ValueError('Require at least two confirmation frames')
        if self.min_confidence > 1 or self.interpolation_gap_s > self.max_gap_s:
            raise ValueError('Invalid confidence or interpolation horizon')


def _valid_candidates(detections, width, height, config):
    valid = []
    for index, d in enumerate(detections):
        if all(k in d and math.isfinite(d[k]) for k in ('x', 'y', 'conf')):
            if 0 <= d['x'] < width and 0 <= d['y'] < height and config.min_confidence <= d['conf'] <= 1:
                valid.append((index, d))
    return valid


def track_temporal(detections, width, height, fps, config=TemporalConfig()):
    """Return one record per frame; never promote an inferred point to a measurement."""
    if not math.isfinite(fps) or fps <= 0 or min(width, height) <= 0:
        raise ValueError('Invalid dimensions or FPS')
    diag = math.hypot(width, height)
    dt = 1.0 / fps
    transition = np.eye(4)
    transition[0, 2] = transition[1, 3] = dt
    observation = np.eye(2, 4)
    noise_map = np.array([[dt*dt/2, 0], [0, dt*dt/2], [dt, 0], [0, dt]])
    process_noise = noise_map @ noise_map.T * (diag*config.acceleration_std_diagonal_s2)**2
    measurement_noise = np.eye(2) * (diag*config.measurement_std_diagonal)**2
    state = covariance = last_point = None
    last_frame = None
    pending = []
    segment = 0
    output = []
    max_gap = max(1, math.floor(config.max_gap_s*fps + 1e-9))
    for frame, raw in enumerate(detections):
        candidates = _valid_candidates(raw, width, height, config)
        record = dict(frame_idx=frame, timestamp_s=frame/fps,
                      state='none', x=None, y=None, conf=None, segment_id=None,
                      position_source='unknown', measurement_xy=None, filtered_xy=None,
                      covariance_xy=None, accepted_candidate_index=None, rejected=[],
                      reason='no_supported_measurement')
        if state is not None and frame-last_frame > max_gap:
            state = covariance = last_point = None
            pending = []
        if state is None:
            # A new segment needs a coherent streak; confidence alone cannot teleport it.
            if pending:
                previous_frame, previous = pending[-1]
                radius = diag*config.max_speed_diagonal_s*(frame-previous_frame)/fps
                eligible = [(index, d) for index, d in candidates
                            if math.hypot(d['x']-previous['x'], d['y']-previous['y']) <= radius]
            else:
                eligible = candidates
            if eligible:
                index, chosen = max(eligible, key=lambda c: c[1]['conf'])
                pending.append((frame, chosen))
            else:
                pending = []
                if candidates:
                    index, chosen = max(candidates, key=lambda c: c[1]['conf'])
                    pending = [(frame, chosen)]
            if len(pending) >= config.confirmation_frames:
                first_frame, first = pending[0]
                elapsed = (frame-first_frame)/fps
                velocity = (np.array([chosen['x'], chosen['y']])-np.array([first['x'], first['y']]))/elapsed
                state = np.array([chosen['x'], chosen['y'], *velocity], dtype=float)
                covariance = np.diag([(diag*config.measurement_std_diagonal)**2]*2 + [(diag*.2)**2]*2)
                last_point = np.array([chosen['x'], chosen['y']])
                last_frame = frame
                segment += 1
                pending = []
                record.update(state='detected', x=float(state[0]), y=float(state[1]),
                              conf=chosen['conf'], segment_id=segment, measurement_xy=last_point.tolist(),
                              filtered_xy=state[:2].tolist(), accepted_candidate_index=index,
                              covariance_xy=covariance[:2, :2].tolist(),
                              position_source='measurement_filtered', reason='confirmed_new_segment')
            elif pending:
                record['reason'] = 'pending_temporal_confirmation'
        else:
            predicted = transition @ state
            prior_covariance = transition @ covariance @ transition.T + process_noise
            innovation_covariance = observation @ prior_covariance @ observation.T + measurement_noise
            ranked = []
            for index, candidate in candidates:
                point = np.array([candidate['x'], candidate['y']])
                residual = point-predicted[:2]
                distance = float(np.linalg.norm(residual))
                motion = float(np.linalg.norm(point-last_point))
                elapsed = (frame-last_frame)/fps
                mahalanobis = float(residual @ np.linalg.solve(innovation_covariance, residual))
                if motion > diag*config.max_speed_diagonal_s*elapsed:
                    reason = 'speed_gate'
                elif distance > diag*config.max_innovation_diagonal:
                    reason = 'innovation_distance_gate'
                elif mahalanobis > config.mahalanobis_squared_gate:
                    reason = 'mahalanobis_gate'
                else:
                    reason = None
                    ranked.append((mahalanobis-.5*candidate['conf'], index, candidate, residual))
                if reason:
                    record['rejected'].append(dict(candidate_index=index, reason=reason,
                                                    innovation_px=distance, mahalanobis_squared=mahalanobis))
            if ranked:
                _, index, chosen, residual = min(ranked, key=lambda item: item[0])
                gain = np.linalg.solve(innovation_covariance, observation @ prior_covariance).T
                state = predicted + gain @ residual
                correction = np.eye(4)-gain @ observation
                covariance = correction @ prior_covariance @ correction.T + gain @ measurement_noise @ gain.T
                last_point = np.array([chosen['x'], chosen['y']])
                last_frame = frame
                record.update(state='detected', conf=chosen['conf'], measurement_xy=last_point.tolist(),
                              accepted_candidate_index=index, position_source='measurement_filtered',
                              reason='accepted_motion_gates')
            else:
                state, covariance = predicted, prior_covariance
                record.update(state='predicted', position_source='kalman_extrapolation',
                              reason='motion_rejected' if candidates else 'missing_measurement')
            if 0 <= state[0] < width and 0 <= state[1] < height:
                record.update(x=float(state[0]), y=float(state[1]), segment_id=segment,
                              filtered_xy=state[:2].tolist(), covariance_xy=covariance[:2, :2].tolist())
            else:
                state = covariance = last_point = None
                pending = []
                record.update(state='none', x=None, y=None, segment_id=None,
                              conf=None, measurement_xy=None, filtered_xy=None,
                              covariance_xy=None, accepted_candidate_index=None,
                              position_source='unknown', reason='prediction_outside_image')
        output.append(record)

    # Offline interpolation uses two accepted endpoints, only inside the same segment.
    maximum = math.floor(config.interpolation_gap_s*fps + 1e-9)
    accepted = [i for i, r in enumerate(output) if r['state']=='detected']
    for left, right in zip(accepted, accepted[1:]):
        a, b = output[left], output[right]
        if not (0 < right-left-1 <= maximum and a['segment_id']==b['segment_id']):
            continue
        gap = output[left+1:right]
        if not all(r['state']=='predicted' and r['segment_id']==a['segment_id'] for r in gap):
            continue
        for index in range(left+1, right):
            weight = (index-left)/(right-left)
            point = (1-weight)*np.array([a['x'], a['y']])+weight*np.array([b['x'], b['y']])
            output[index].update(state='interpolated', x=float(point[0]), y=float(point[1]),
                                 conf=None, measurement_xy=None, covariance_xy=None,
                                 position_source='offline_linear_interpolation', reason='bounded_gap_between_measurements',
                                 interpolation_endpoints=[left, right])
    return output


def trail_segments(tracked, index, length=25):
    """Only connect consecutive accepted measurements in the same segment."""
    lines, current = [], []
    previous = None
    for i in range(max(0, index-length), index+1):
        point = tracked[i]
        if point['state'] != 'detected' or (previous is not None and
                point.get('segment_id') != previous.get('segment_id')):
            if len(current)>1:
                lines.append(current)
            current = []
        if point['state']=='detected':
            current.append((point['x'], point['y']))
        previous = point
    if len(current)>1:
        lines.append(current)
    return lines

def read_video(path):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = []
    while True:
        ok, img = cap.read()
        if not ok:
            break
        frames.append(img)
    cap.release()
    return frames, fps

def write_video(path, frames, fps, frame_shape=None):
    h, w = frames[0].shape[:2] if frame_shape is None else frame_shape
    proc = subprocess.Popen(['ffmpeg', '-v', 'error', '-y', '-f', 'rawvideo', '-pix_fmt', 'bgr24',
                             '-s', f'{w}x{h}', '-r', str(fps), '-i', '-', '-an', '-c:v', 'libx264',
                             '-crf', '23', '-pix_fmt', 'yuv420p', '-movflags', '+faststart', str(path)],
                            stdin=subprocess.PIPE)
    for img in frames:
        proc.stdin.write(np.ascontiguousarray(img).tobytes())
    proc.stdin.close()
    assert proc.wait() == 0, f'ffmpeg failed for {path}'

def show(path, width=960):
    display(Video(str(path), embed=True, width=width, html_attributes='controls loop'))

def merge(dets, radius):
    # Drop duplicates (same ball found twice) by center distance; keep the most confident.
    kept = []
    for d in sorted(dets, key=lambda d: -d['conf']):
        if all(math.hypot(d['x'] - k['x'], d['y'] - k['y']) > radius for k in kept):
            kept.append(d)
    return kept

def track(dets_per_frame, width, height, fps, min_conf, max_gap=6):
    # One ball, constant-velocity prediction, gated nearest/most-confident association.
    diag = math.hypot(width, height)
    pos = vel = None
    last = None
    segment = 0
    out = []
    for i, dets in enumerate(dets_per_frame):
        cands = [d for d in dets if d['conf'] >= min_conf]
        if pos is not None and i - last > max_gap:
            pos = None
        chosen, reinit = None, False
        if pos is not None:
            gap = i - last
            pred = pos + vel * gap
            gate = diag * (0.03 + 0.04 * gap)
            scored = []
            for d in cands:
                dist = math.hypot(d['x'] - pred[0], d['y'] - pred[1])
                if dist <= gate:
                    scored.append((d['conf'] - 0.5 * dist / gate, d))
            if scored:
                chosen = max(scored, key=lambda s: s[0])[1]
            elif cands and max(c['conf'] for c in cands) >= 0.6:
                chosen, reinit = max(cands, key=lambda d: d['conf']), True
        elif cands:
            chosen, reinit = max(cands, key=lambda d: d['conf']), True
        if chosen is not None:
            if reinit:
                segment += 1
            p = np.array([chosen['x'], chosen['y']])
            vel = np.zeros(2) if reinit else 0.6 * vel + 0.4 * (p - pos) / (i - last)
            pos, last = p, i
            out.append(dict(state='detected', x=float(p[0]), y=float(p[1]), conf=chosen['conf'], segment_id=segment))
        elif pos is not None:
            pred = pos + vel * (i - last)
            if 0 <= pred[0] < width and 0 <= pred[1] < height:
                out.append(dict(state='predicted', x=float(pred[0]), y=float(pred[1]), conf=None, segment_id=segment))
            else:
                pos = None
                out.append(dict(state='none'))
        else:
            out.append(dict(state='none'))
    return out

def draw(img, dets, tracked, index, label, scale=1.0):
    img = img.copy() if scale == 1.0 else cv2.resize(img, None, fx=scale, fy=scale)
    u = max(1, round(img.shape[0] / 360))
    if SHOW_RAW_CANDIDATES:
        for d in dets[index]:
            r = max(5 * u, int(d.get('r', 0) * scale) + 3 * u)
            cv2.circle(img, (int(d['x'] * scale), int(d['y'] * scale)), r, (0, 140, 255), u)
    for segment_points in trail_segments(tracked, index):
        points = np.array([(int(x*scale), int(y*scale)) for x, y in segment_points], np.int32)
        cv2.polylines(img, [points], False, (0, 200, 0), u, cv2.LINE_AA)
    t = tracked[index]
    if t['state'] == 'detected':
        c = (int(t['x'] * scale), int(t['y'] * scale))
        cv2.circle(img, c, 11 * u, (0, 230, 0), 2 * u)
        cv2.putText(img, f"{t['conf']:.2f}", (c[0] + 12 * u, c[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.4 * u,
                    (0, 230, 0), u, cv2.LINE_AA)
    elif t['state'] == 'interpolated':
        c = (int(t['x']*scale), int(t['y']*scale))
        cv2.circle(img, c, 9*u, (255, 210, 0), u)
        cv2.putText(img, 'inferred', (c[0]+10*u, c[1]), cv2.FONT_HERSHEY_SIMPLEX, .35*u, (255, 210, 0), u)
    elif t['state'] == 'predicted':
        x, y, s = int(t['x'] * scale), int(t['y'] * scale), 6 * u
        cv2.line(img, (x - s, y - s), (x + s, y + s), (170, 170, 170), u)
        cv2.line(img, (x - s, y + s), (x + s, y - s), (170, 170, 170), u)
    cv2.rectangle(img, (0, 0), (img.shape[1], 16 * u), (0, 0, 0), -1)
    cv2.putText(img, f"{label} | frame {index} | {t['state']}", (4 * u, 12 * u), cv2.FONT_HERSHEY_SIMPLEX,
                0.4 * u, (255, 255, 255), u, cv2.LINE_AA)
    return img

# Draft check points: ball centers marked by eye on 19 frames (1280x720 coordinates).
# Not certified ground truth; ambiguous frames were left out.
REFERENCE = {
    'attack_clip': {0: (184, 384), 30: (166, 410), 40: (130, 426), 50: (174, 312), 80: (330, 358),
                    90: (348, 358), 100: (364, 218), 120: (685, 95)},
    'attack_clip2': {0: (251, 476), 24: (337, 474), 36: (440, 495), 48: (483, 471), 60: (542, 483),
                     96: (783, 572), 108: (744, 460), 120: (650, 392), 144: (695, 407), 156: (730, 337),
                     168: (765, 229)},
}
MATCH_RADIUS = 14  # pixels in 1280x720 space
(OUT / 'evaluation_draft.json').write_text(json.dumps(dict(
    reference=REFERENCE, coordinate_size=[1280, 720], match_radius=MATCH_RADIUS,
    status='draft development checkpoints, not held-out ground truth'), indent=2))

def score(clip, dets, tracked, width, height):
    sx, sy = 1280 / width, 720 / height
    raw_hits = tracked_hits = tracked_wrong = false_candidates = candidates = 0
    for frame, (rx, ry) in REFERENCE[clip].items():
        near = lambda d: math.hypot(d['x'] * sx - rx, d['y'] * sy - ry) <= MATCH_RADIUS
        raw_hits += any(near(d) for d in dets[frame])
        candidates += len(dets[frame])
        false_candidates += sum(not near(d) for d in dets[frame])
        t = tracked[frame]
        if t['state'] == 'detected':
            if near(t):
                tracked_hits += 1
            else:
                tracked_wrong += 1
    return dict(check_frames=len(REFERENCE[clip]), raw_found=raw_hits, tracked_correct=tracked_hits,
                tracked_wrong_place=tracked_wrong, check_candidates=candidates,
                check_off_target_candidates=false_candidates)

RESULTS = {}

def run_model(label, detect, tracking=None):
    # detect(clip_name, frames) -> (list of per-frame detections [{x, y, r, conf}], seconds)
    RESULTS[label] = {}
    for clip, path in CLIPS.items():
        try:
            frames, fps = read_video(path)
            h, w = frames[0].shape[:2]
            dets, seconds = detect(clip, frames)
            assert len(dets) == len(frames), 'detector returned wrong frame count'
            tracking_started = time.perf_counter()
            tracked = (tracking or track)(dets, w, h, fps, MIN_CONF[label])
            tracking_seconds = time.perf_counter()-tracking_started
            video = OUT / f'{label}_{clip}.mp4'
            write_video(video, (draw(f, dets, tracked, i, label) for i, f in enumerate(frames)),
                        fps, frame_shape=(h, w))
            states = [t['state'] for t in tracked]
            stats = dict(frames=len(frames), detector_seconds=round(seconds, 3),
                         fps_model=round(len(frames) / seconds, 1),
                         detector_time_origin='cached_original_run' if RUN_CONFIGS.get(label, {}).get('reused_detections') else 'this_run',
                         frames_with_any_detection=sum(bool(d) for d in dets),
                         frames_ball_tracked=states.count('detected'), frames_predicted=states.count('predicted'),
                         frames_interpolated=states.count('interpolated'), tracker_seconds=round(tracking_seconds, 4),
                         motion_rejections=sum(len(t.get('rejected', [])) for t in tracked),
                         **score(clip, dets, tracked, w, h))
            RESULTS[label][clip] = dict(stats=stats, detections=dets, tracked=tracked)
            (OUT / f'{label}_{clip}.json').write_text(json.dumps(dict(stats=stats, detections=dets,
                tracked=tracked, config=RUN_CONFIGS.get(label, {}))))
            print(label, clip, stats)
            if SHOW_BASELINE or not COMPARE_KALMAN or label.endswith('-Kalman'):
                show(video)
        except Exception as error:
            RESULTS[label][clip] = dict(error=f'{type(error).__name__}: {error}')
            print(label, clip, 'FAILED:', RESULTS[label][clip]['error'])
            (OUT / f'{label}_{clip}_error.json').write_text(json.dumps(RESULTS[label][clip]))

## 1. WASB-SBDT: low-resolution sweep (same basketball weights)
Official repo + authors' basketball checkpoint (cached in `MyDrive/TacticalAI/models/`).
Each step feeds 3 consecutive frames at 512x288 and gets 3 heatmaps back, so every frame
is checked up to 3 times.

Default `lowres` profile, **clip 1 only**:

| Variant | Views | Threshold | Change from reference |
|---|---|---|---|
| WASB-tiles-020 | full + 2x2 crops (60% size) | 0.20 | Previous final configuration |
| WASB-tight-020 | full + 3x3 crops (40% size) | 0.20 | Tighter crops |
| WASB-tight-010 | full + 3x3 crops (40% size) | 0.10 | Tighter crops, lower threshold |
| WASB-tight-cubic | full + 3x3 crops (40% size) | 0.20 | Tighter crops, cubic resizing |
| WASB-tight-clahe | full + 3x3 crops (40% size) | 0.20 | Tighter crops, mild luminance CLAHE |

At 640x360, 40% crops are 256x144, resized to 512x288. The ball occupies
more network pixels, but resizing does not recover lost image detail. CLAHE
uses clipLimit=1.5, grid=8x8 on LAB luminance before resizing. This is an
experiment, not a proven improvement; contrast may also amplify compression noise.
All crops cover the whole image automatically; no hand-marked locations guide inference.
The full frame is retained for context and boundary recovery. Distinct views are shared
between thresholds but processed again for different interpolation/preprocessing.
This profile uses 34 distinct view/preprocessing combinations per temporal window,
versus five for the previous tiled detector, so expect substantially more GPU work.
Visible tiny-ball misses and actual occlusion are different failures: no hidden-ball
measurements are created. Gray crosses remain short motion predictions, then unknown.

`original` profile retains the previous settings on both clips:

| Variant | Views | Heatmap / tracker threshold |
|---|---|---|
| WASB-base | full frame | 0.50 / 0.50 (original baseline) |
| WASB-full-035 | full frame | 0.35 / 0.35 |
| WASB-full-020 | full frame | 0.20 / 0.20 |
| WASB-full-010 | full frame | 0.10 / 0.10 |
| WASB-tiles-035 | full + four overlapping 60% crops | 0.35 / 0.35 |
| WASB-tiles-020 | full + four overlapping 60% crops | 0.20 / 0.20 |

Only inference/postprocessing changes; no retraining. Crop coordinates are mapped back
automatically. Threshold variants reuse the same heatmaps. Reported detector time
includes preprocessing, GPU inference/transfers, and candidate extraction/merging;
shared inference time is charged to each applicable variant, not measured as isolated runs.
It excludes video decoding, tracking and export. Standard tiles cost five views per
window; tighter crops cost ten, before sharing identical views between variants.
The tracker retains the original 0.6 out-of-gate reinitialization rule and six-frame gap.
Lower thresholds can produce many false positives. Draft checkpoints are development
feedback used to motivate this sweep; they are not an independent evaluation set.

In [ ]:
if RUN_WASB:
    import importlib.util, gdown
    from omegaconf import OmegaConf

    WASB_DIR = Path('/content/WASB-SBDT')
    if not WASB_DIR.exists():
        subprocess.run(['git', 'clone', '-q', 'https://github.com/nttcom/WASB-SBDT.git', str(WASB_DIR)], check=True)
    WASB_COMMIT = '923462cacdeb3353b84ddebdedb3f4b7a8553b0f'
    subprocess.run(['git', '-C', str(WASB_DIR), 'checkout', '-q', WASB_COMMIT], check=True)
    WASB_WEIGHTS = MODELS_DIR / 'wasb_basketball_best.pth.tar'
    if not WASB_WEIGHTS.exists():
        gdown.download(id='1nfECuSyJvPUmz3njZCdFERSQQbERt8FU', output=str(WASB_WEIGHTS), quiet=False)

    spec = importlib.util.spec_from_file_location('wasb_hrnet', WASB_DIR / 'src/models/hrnet.py')
    hrnet = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(hrnet)
    wasb_cfg = OmegaConf.load(WASB_DIR / 'src/configs/model/wasb.yaml')
    wasb = hrnet.HRNet(wasb_cfg)
    state = torch.load(WASB_WEIGHTS, map_location='cpu', weights_only=False)
    state = state.get('model_state_dict', state)
    wasb.load_state_dict({k.removeprefix('module.'): v for k, v in state.items()})
    wasb = wasb.cuda().eval()
    """WASB inference sweep; embedded in the standalone Colab notebook by its builder.

    No labels or manually selected ball locations enter inference. All variants use
    the same checkpoint, consecutive three-frame windows and 512x288 network input.
    """
    import math
    import time

    import cv2
    import numpy as np


    WASB_VARIANTS = {
        'WASB-base': dict(mode='full', threshold=0.50),
        'WASB-full-035': dict(mode='full', threshold=0.35),
        'WASB-full-020': dict(mode='full', threshold=0.20),
        'WASB-full-010': dict(mode='full', threshold=0.10),
        'WASB-tiles-035': dict(mode='full+tiles', threshold=0.35),
        'WASB-tiles-020': dict(mode='full+tiles', threshold=0.20),
    }

    WASB_LOWRES_VARIANTS = {
        'WASB-tiles-020': dict(mode='full+tiles', threshold=0.20, crop_fraction=0.6),
        'WASB-tight-020': dict(mode='full+tiles', threshold=0.20, crop_fraction=0.4),
        'WASB-tight-010': dict(mode='full+tiles', threshold=0.10, crop_fraction=0.4),
        'WASB-tight-cubic': dict(mode='full+tiles', threshold=0.20, crop_fraction=0.4,
                                interpolation='cubic'),
        'WASB-tight-clahe': dict(mode='full+tiles', threshold=0.20, crop_fraction=0.4,
                                preprocessing='clahe'),
    }


    def wasb_views(width, height, tiled=False, crop_fraction=0.6):
        if not 0 < crop_fraction <= 1:
            raise ValueError('crop_fraction must be in (0, 1]')
        views = [(0, 0, width, height)]
        if tiled:
            cw, ch = max(1, round(width * crop_fraction)), max(1, round(height * crop_fraction))
            # At least 20% overlap. 60% crops give 2x2; 40% crops give 3x3.
            nx = math.ceil((width-cw)/max(1, cw*0.8))+1
            ny = math.ceil((height-ch)/max(1, ch*0.8))+1
            xs = np.linspace(0, width-cw, nx).round().astype(int)
            ys = np.linspace(0, height-ch, ny).round().astype(int)
            views += [(int(x), int(y), cw, ch) for y in ys for x in xs]
        return list(dict.fromkeys(views))


    def wasb_preprocess(image, method='none'):
        if method == 'none':
            return image
        if method != 'clahe':
            raise ValueError(f'Unknown preprocessing: {method}')
        lab = cv2.cvtColor(image, cv2.COLOR_BGR2LAB)
        clahe = cv2.createCLAHE(clipLimit=1.5, tileGridSize=(8, 8))
        lab[:, :, 0] = clahe.apply(lab[:, :, 0])
        return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


    def wasb_plan(width, height, variants):
        """Share inference only when view, preprocessing and interpolation match."""
        plan = {}
        for name, config in variants.items():
            views = wasb_views(width, height, config['mode'] == 'full+tiles',
                               config.get('crop_fraction', 0.6))
            for view in views:
                key = (view, config.get('preprocessing', 'none'), config.get('interpolation', 'linear'))
                plan.setdefault(key, []).append(name)
        return plan


    def wasb_components(hm, threshold, view):
        """Map connected heatmap components back to source-image coordinates."""
        ox, oy, width, height = view
        scale = min(512 / width, 288 / height)
        tx, ty = (512-scale*width)/2, (288-scale*height)/2
        count, labels = cv2.connectedComponents((hm > threshold).astype(np.uint8))
        found = []
        for lab in range(1, count):
            ys, xs = np.nonzero(labels == lab)
            weights = hm[ys, xs]
            x = ((xs*weights).sum()/weights.sum()-tx)/scale
            y = ((ys*weights).sum()/weights.sum()-ty)/scale
            if 0 <= x < width and 0 <= y < height:
                found.append(dict(x=float(x+ox), y=float(y+oy),
                                  r=float(max(np.ptp(xs)+1, np.ptp(ys)+1)/scale/2),
                                  conf=float(weights.max())))
        return found


    def wasb_merge(detections, radius):
        kept = []
        for d in sorted(detections, key=lambda d: -d['conf']):
            if all(math.hypot(d['x']-k['x'], d['y']-k['y']) > radius for k in kept):
                kept.append(d)
        return kept


    def wasb_sweep(model, frames, batch=4, variants=None):
        """Run each view once, reuse its heatmaps across threshold variants.

        Report inference/preprocessing/transfer time shared by threshold variants,
        plus each variant's extraction and merge time. Excludes decode/tracking/export.
        Small batches limit Colab T4 memory; no full-clip heatmap cache is retained.
        """
        import torch
        variants = WASB_VARIANTS if variants is None else variants
        if len(frames) < 3:
            raise ValueError('WASB needs at least three frames')
        height, width = frames[0].shape[:2]
        found = {name: [[] for _ in frames] for name in variants}
        extraction = {name: 0.0 for name in variants}
        inference_seconds = {name: 0.0 for name in variants}
        mean = np.array([0.485, 0.456, 0.406], np.float32)
        std = np.array([0.229, 0.224, 0.225], np.float32)
        plan = wasb_plan(width, height, variants)
        for vi, ((view, preprocessing, interpolation), names) in enumerate(plan.items(), 1):
            print(f'View {vi}/{len(plan)}: {view}, {preprocessing}, {interpolation}', flush=True)
            torch.cuda.synchronize()
            t0 = time.perf_counter()
            ox, oy, cw, ch = view
            scale = min(512/cw, 288/ch)
            matrix = np.float32([[scale, 0, (512-scale*cw)/2], [0, scale, (288-scale*ch)/2]])
            inputs = []
            for f in frames:
                image = wasb_preprocess(f[oy:oy+ch, ox:ox+cw], preprocessing)
                image = cv2.warpAffine(image, matrix, (512, 288), flags={
                    'linear': cv2.INTER_LINEAR, 'cubic': cv2.INTER_CUBIC}[interpolation])
                rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB).astype(np.float32)/255
                inputs.append(torch.from_numpy(((rgb-mean)/std).transpose(2, 0, 1)))
            elapsed = time.perf_counter()-t0
            active = {name: variants[name] for name in names}
            with torch.inference_mode():
                for start in range(0, len(frames)-2, batch):
                    indices = list(range(start, min(start+batch, len(frames)-2)))
                    torch.cuda.synchronize()
                    t0 = time.perf_counter()
                    inp = torch.stack([torch.cat(inputs[i:i+3]) for i in indices]).cuda()
                    maps = model(inp)[0].sigmoid().cpu().numpy()
                    torch.cuda.synchronize()
                    elapsed += time.perf_counter()-t0
                    for name, config in active.items():
                        t0 = time.perf_counter()
                        for i, heatmaps in zip(indices, maps):
                            for k, hm in enumerate(heatmaps):
                                found[name][i+k].extend(wasb_components(hm, config['threshold'], view))
                        extraction[name] += time.perf_counter()-t0
            for name in names:
                inference_seconds[name] += elapsed
            del inputs
        results = {}
        for name, config in variants.items():
            t0 = time.perf_counter()
            # Preserve original baseline merge radius; use crop scale for tiled variants.
            scale = min(512/width, 288/height)
            radius = 4/scale * (config.get('crop_fraction', 0.6) if config['mode'] == 'full+tiles' else 1)
            detections = [wasb_merge(d, radius) for d in found[name]]
            extraction[name] += time.perf_counter()-t0
            results[name] = (detections, inference_seconds[name]+extraction[name])
        return results


    ACTIVE_VARIANTS = ({'WASB-tiles-020': WASB_VARIANTS['WASB-tiles-020']}
                       if EXPERIMENT == 'rfdet' else
                       WASB_LOWRES_VARIANTS if EXPERIMENT == 'lowres' else WASB_VARIANTS)
    weight_sha256 = hashlib.sha256(WASB_WEIGHTS.read_bytes()).hexdigest()
    for label, raw_config in ACTIVE_VARIANTS.items():
        config = dict(raw_config)
        config.setdefault('crop_fraction', 0.6)
        config.setdefault('preprocessing', 'none')
        config.setdefault('interpolation', 'linear')
        MIN_CONF[label] = config['threshold']
        RUN_CONFIGS[label] = dict(**config, tracker_min_conf=config['threshold'],
            tracker_max_gap=6, tracker_reinitialize_conf=0.6, batch=4,
            checkpoint_sha256=weight_sha256, upstream_commit=WASB_COMMIT,
            input_size=[512, 288], frames_in=3, frames_out=3,
            clahe_clip_limit=1.5 if config['preprocessing'] == 'clahe' else None,
            clahe_grid=[8, 8] if config['preprocessing'] == 'clahe' else None,
            experiment=EXPERIMENT,
            reid=False, gmc=False, offline=True,
            timing='shared inference + variant extraction; excludes decode/tracking/export')
    (OUT / 'wasb_configs.json').write_text(json.dumps(RUN_CONFIGS, indent=2))

    # Cache candidate lists, not heatmaps. Each clip is inferred once per view.
    sweep_cache = {}
    sweep_errors = {}
    def variant_detector(label):
        def detect(clip, frames):
            if clip in sweep_errors:
                raise RuntimeError(sweep_errors[clip])
            if clip not in sweep_cache:
                print('Inferring selected WASB views for', clip, EXPERIMENT, flush=True)
                try:
                    sweep_cache[clip] = wasb_sweep(wasb, frames, batch=4, variants=ACTIVE_VARIANTS)
                except Exception as error:
                    sweep_errors[clip] = str(error)
                    raise
            return sweep_cache[clip][label]
        return detect

    for label in ACTIVE_VARIANTS:
        print('\nVARIANT', label, RUN_CONFIGS[label]['mode'], 'threshold', MIN_CONF[label])
        run_model(label, variant_detector(label))
    del wasb; torch.cuda.empty_cache()
else:
    print('WASB skipped (RUN_WASB = False; previously checked).')


## 2. RF-DETR basketball (Roboflow)
Runs in its own small Python environment (avoids package clashes with this notebook).
Weights download with your key; inference runs locally in Colab. The default run selects **RF-DETR-tiles on both clips**, with the Kalman overlay. The available variants are:

| Variant | Views | Candidate cutoff | Tracker cutoff |
|---|---|---|---|
| RF-DETR-full | full frame | 0.10 | 0.25 |
| RF-DETR-tiles | full + 2x2, 60% crops (baseline) | 0.10 | 0.25 |
| RF-DETR-tight | full + 3x3, 40% crops | 0.10 | 0.25 |
| RF-DETR-tight-low | same tighter crops | 0.05 | 0.25 |
| RF-DETR-tight-track010 | same tighter crops | 0.05 | 0.10 |
| RF-DETR-tiles-track040 | baseline views | 0.10 | 0.40 |

Tight variants share inference at 0.05, then filter candidates to their stated cutoff.
Baseline variants keep inference at 0.10. No checkpoint location enters detection or
tracking. Crop batches are limited to four views. Timings charge shared inference
to each applicable variant, not independently timed runs. Tracking gap (6 frames)
and reinitialization confidence (0.6) stay fixed. Raw checkpoint hits, wrong tracked
locations, and off-target candidates matter more than visual coverage.
These tests were selected after reviewing the previous run: only 3/8 draft hits,
with 9 off-target candidates for full frame and 24 for tiles. This is development
feedback, not an independent evaluation. Set `RF_SWEEP = False` for the two baselines.
The adapter's underlying architecture, active execution provider, and library versions are saved.
A three-frame full/tiled smoke test must pass on GPU before the full clip runs.
An isolated Python 3.11 / Torch CUDA 12.4 / ONNX Runtime CUDA 12 environment
avoids inheriting Colab's CUDA 13 libraries. Its first installation can take several minutes.
Model availability and access are checked when your key is supplied; this checkpoint has
not yet been downloaded or executed here. No cloud inference fallback is used.

In [ ]:
RF_SCRIPT = Path('/content/rf_detect.py')
RF_SCRIPT.write_text('"""Local Roboflow basketball inference, embedded into the standalone notebook."""\nimport json\nimport math\nimport os\nimport sys\nimport time\nfrom importlib import metadata\n\nimport cv2\n\n\nBALL_CLASSES = {\'ball\', \'ball-in-basket\'}\n\n\ndef model_runtime(model):\n    """Validate the real detector, including Roboflow\'s generic adapter."""\n    backend = getattr(model, \'_model\', model)\n    architecture = type(backend).__name__\n    if \'rfdetr\' not in architecture.lower().replace(\'_\', \'\').replace(\'-\', \'\'):\n        raise RuntimeError(f\'Expected RF-DETR backend; got {architecture}. Check RF_MODEL_ID.\')\n    session = getattr(backend, \'_session\', None)\n    if session is None:\n        session = getattr(backend, \'session\', None)\n    providers = list(session.get_providers()) if session is not None else []\n    device = str(getattr(backend, \'_device\', getattr(backend, \'device\', \'unknown\')))\n    if session is not None:\n        if \'CUDAExecutionProvider\' not in providers:\n            raise RuntimeError(f\'RF-DETR fell back to CPU: {providers}. Repair the isolated CUDA environment.\')\n    elif not device.startswith(\'cuda\'):\n        raise RuntimeError(f\'Cannot verify RF-DETR GPU execution (device={device}).\')\n    return dict(architecture_handler=type(model).__name__, backend_class=architecture,\n                backend_module=type(backend).__module__, execution_providers=providers,\n                device=device)\n\n\ndef ball_candidates(predictions, views):\n    if len(predictions) != len(views):\n        raise ValueError(\'Inference returned a different number of views\')\n    detections, names = [], set()\n    for response, (ox, oy, _, _) in zip(predictions, views):\n        for d in response.predictions:\n            name = d.class_name.strip().lower()\n            names.add(name)\n            if name in BALL_CLASSES:\n                detections.append(dict(x=float(d.x+ox), y=float(d.y+oy),\n                    r=float(max(d.width, d.height)/2), conf=float(d.confidence), cls=name))\n    return detections, names\n\n\ndef merge(detections):\n    kept = []\n    for d in sorted(detections, key=lambda d: -d[\'conf\']):\n        if all(math.hypot(d[\'x\']-k[\'x\'], d[\'y\']-k[\'y\']) > max(4, d[\'r\']) for k in kept):\n            kept.append(d)\n    return kept\n\n\n# One change at a time around the existing full-frame/2x2 baselines.\nRF_VARIANTS = {\n    \'RF-DETR-full\': dict(mode=\'full\', candidate_confidence=0.10, tracker_min_conf=0.25),\n    \'RF-DETR-tiles\': dict(mode=\'tiles\', candidate_confidence=0.10, tracker_min_conf=0.25),\n    \'RF-DETR-tight\': dict(mode=\'tight\', candidate_confidence=0.10, tracker_min_conf=0.25),\n    \'RF-DETR-tight-low\': dict(mode=\'tight\', candidate_confidence=0.05, tracker_min_conf=0.25),\n    \'RF-DETR-tight-track010\': dict(mode=\'tight\', candidate_confidence=0.05, tracker_min_conf=0.10),\n    \'RF-DETR-tiles-track040\': dict(mode=\'tiles\', candidate_confidence=0.10, tracker_min_conf=0.40),\n}\n\n\ndef crop_views(width, height, mode):\n    if mode == \'full\':\n        return [(0, 0, width, height)]\n    grid, fraction = (2, 0.6) if mode == \'tiles\' else (3, 0.4)\n    cw, ch = max(1, round(width*fraction)), max(1, round(height*fraction))\n    return [(round(x*(width-cw)/(grid-1)), round(y*(height-ch)/(grid-1)), cw, ch)\n            for y in range(grid) for x in range(grid)]\n\n\ndef detect_clip(model, frames, sweep=False, variant_labels=None):\n    height, width = frames[0].shape[:2]\n    variants = RF_VARIANTS if sweep else {k: RF_VARIANTS[k] for k in (\'RF-DETR-full\', \'RF-DETR-tiles\')}\n    if variant_labels is not None:\n        if not variant_labels or any(label not in RF_VARIANTS for label in variant_labels):\n            raise ValueError(\'Select at least one known RF-DETR variant.\')\n        variants = {label: RF_VARIANTS[label] for label in variant_labels}\n    results = {label: [] for label in variants}\n    seconds = {label: 0.0 for label in variants}\n    names = set()\n    full = crop_views(width, height, \'full\')\n    tiles = crop_views(width, height, \'tiles\')\n    tight = crop_views(width, height, \'tight\')\n    for i, image in enumerate(frames):\n        cache = {}\n\n        def infer_views(key, views, threshold):\n            t0 = time.perf_counter()\n            crops = [image[y:y+h, x:x+w] for x, y, w, h in views]\n            # Keep crop batches small enough for a T4.\n            candidates = []\n            for start in range(0, len(crops), 4):\n                stop = start+4\n                ds, found_names = ball_candidates(\n                    model.infer(crops[start:stop], confidence=threshold), views[start:stop])\n                candidates.extend(ds)\n                names.update(found_names)\n            cache[key] = (candidates, time.perf_counter()-t0)\n\n        modes = {config[\'mode\'] for config in variants.values()}\n        if modes & {\'full\', \'tiles\'}:\n            infer_views(\'full\', full, 0.10)\n        if \'tiles\' in modes:\n            infer_views(\'tiles\', tiles, 0.10)\n        if \'tight\' in modes:\n            infer_views(\'full-low\', full, 0.05)\n            infer_views(\'tight\', tight, 0.05)\n        for label, config in variants.items():\n            keys = {\'full\': [\'full\'], \'tiles\': [\'full\', \'tiles\'],\n                    \'tight\': [\'full-low\', \'tight\']}[config[\'mode\']]\n            t0 = time.perf_counter()\n            ds = [d for key in keys for d in cache[key][0]\n                  if d[\'conf\'] >= config[\'candidate_confidence\']]\n            # Same merge order as the original tiled baseline.\n            if config[\'mode\'] != \'full\':\n                ds = merge([d for d in cache[keys[0]][0]\n                            if d[\'conf\'] >= config[\'candidate_confidence\']]) + [\n                    d for d in cache[keys[1]][0] if d[\'conf\'] >= config[\'candidate_confidence\']]\n            results[label].append(merge(ds))\n            seconds[label] += sum(cache[key][1] for key in keys) + time.perf_counter()-t0\n        if i % 25 == 0:\n            print(f\'RF-DETR: {i+1}/{len(frames)} frames\', flush=True)\n    return dict(variants={label: dict(detections=results[label], seconds=seconds[label],\n                                    config=dict(config,\n                                        inference_confidence=0.05 if config[\'mode\']==\'tight\' else 0.10,\n                                        crop_fraction={\'full\': 1.0, \'tiles\': 0.6, \'tight\': 0.4}[config[\'mode\']],\n                                        tile_grid={\'full\': [1, 1], \'tiles\': [2, 2], \'tight\': [3, 3]}[config[\'mode\']]))\n                          for label, config in variants.items()},\n                class_names=sorted(names))\n\n\ndef main():\n    # Load CUDA 12/cuDNN from the isolated environment before creating any session.\n    import torch\n    import onnxruntime as ort\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'RF-DETR environment cannot access the Colab GPU.\')\n    ort.preload_dlls(directory=\'\')\n    from inference import get_model\n    model_id = os.environ[\'RF_MODEL_ID\']\n    sweep = os.environ.get(\'RF_SWEEP\', \'false\').lower() == \'true\'\n    selection = os.environ.get(\'RF_VARIANTS\', \'\').strip()\n    variant_labels = selection.split(\',\') if selection else None\n    model = get_model(model_id=model_id, api_key=os.environ[\'ROBOFLOW_API_KEY\'])\n    runtime = model_runtime(model)\n    print(\'Requested model:\', model_id, \'| Runtime:\', runtime, flush=True)\n    versions = {}\n    for package in (\'inference-gpu\', \'inference-models\', \'torch\', \'torchvision\', \'numpy\',\n                    \'opencv-python\', \'onnxruntime-gpu\', \'onnxruntime\', \'nvidia-cublas-cu12\',\n                    \'nvidia-cudnn-cu12\', \'nvidia-cuda-runtime-cu12\'):\n        try:\n            versions[package] = metadata.version(package)\n        except metadata.PackageNotFoundError:\n            versions[package] = None\n    for video, out in zip(sys.argv[1::2], sys.argv[2::2]):\n        # Exercise full-frame AND tiled inference before decoding/processing the full clip.\n        cap = cv2.VideoCapture(video)\n        smoke_frames = []\n        for _ in range(3):\n            ok, image = cap.read()\n            if not ok:\n                break\n            smoke_frames.append(image)\n        cap.release()\n        if not smoke_frames:\n            raise ValueError(f\'Could not decode {video}\')\n        smoke = detect_clip(model, smoke_frames, sweep=sweep, variant_labels=variant_labels)\n        runtime = model_runtime(model)\n        smoke[\'provenance\'] = dict(model_id=model_id, **runtime, packages=versions,\n                                   smoke_frames=len(smoke_frames), cuda=torch.version.cuda,\n                                   gpu=torch.cuda.get_device_name(0))\n        with open(out + \'.smoke.json\', \'w\') as handle:\n            json.dump(smoke, handle)\n        print(\'Three-frame full/tiled smoke test passed; starting full clip.\', flush=True)\n        cap = cv2.VideoCapture(video)\n        frames = []\n        while True:\n            ok, image = cap.read()\n            if not ok:\n                break\n            frames.append(image)\n        cap.release()\n        if not frames:\n            raise ValueError(f\'Could not decode {video}\')\n        data = detect_clip(model, frames, sweep=sweep, variant_labels=variant_labels)\n        data[\'provenance\'] = dict(model_id=model_id, **runtime,\n            cuda=torch.version.cuda, gpu=torch.cuda.get_device_name(0), smoke_frames=len(smoke_frames),\n            packages=versions, sweep=sweep, selected_variants=variant_labels,\n            full_frame=True, local_inference=True,\n            timing_shared_views=True,\n            timing=\'preprocessing/inference/extraction; excludes decode/model load/tracking/export\')\n        with open(out, \'w\') as handle:\n            json.dump(data, handle)\n        print(\'Observed classes:\', data[\'class_names\'], flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n')
RF_LABELS = ['RF-DETR-full', 'RF-DETR-tiles']
if RF_SELECTED_VARIANTS is not None:
    RF_LABELS = list(RF_SELECTED_VARIANTS)
elif RF_SWEEP:
    RF_LABELS += ['RF-DETR-tight', 'RF-DETR-tight-low', 'RF-DETR-tight-track010', 'RF-DETR-tiles-track040']


def rf_detector(source_label, output_label, temporal=False):
    def detect(clip, frames):
        data = json.loads((OUT / f'rf_raw_{clip}.json').read_text())
        result = data['variants'][source_label]
        assert len(result['detections']) == len(frames), 'Cached detection frame count differs from video'
        RUN_CONFIGS[output_label] = dict(**data['provenance'], **result['config'],
            tracker='kalman_cv_gated_interpolation' if temporal else 'legacy_constant_velocity',
            reid=False, gmc=False, offline=True,
            temporal_config=asdict(TemporalConfig(min_confidence=MIN_CONF[output_label], **KALMAN_SETTINGS)) if temporal else None,
            source_detection_file=str(RF_CACHE_SOURCES[clip]) if clip in RF_CACHE_SOURCES else None,
            source_hash_verified=False if clip in RF_CACHE_SOURCES else None,
            reused_detections=clip in RF_CACHE_SOURCES, raw_candidates_visible=SHOW_RAW_CANDIDATES)
        return result['detections'], result['seconds']
    return detect


if (EXPERIMENT == 'rfdet' or RUN_OPTIONAL_MODELS) and (ROBOFLOW_API_KEY or RF_CACHE_AVAILABLE):
    try:
        for clip, source in RF_CACHE_SOURCES.items():
            print('Reusing saved detections:', source)
            data = json.loads(source.read_text())
            assert data['provenance']['model_id'] == RF_MODEL_ID, 'Cached checkpoint differs'
            assert all(label in data['variants'] for label in RF_LABELS), 'Selected variant absent from cache'
            (OUT / f'rf_raw_{clip}.json').write_text(json.dumps(data))
        if RF_CACHE_SOURCES:
            print('Cached model/variant/frame count checked; older caches have no source-video hash.')
        if RF_MISSING_CLIPS:
            venv = Path('/content/venv_rf_cuda12_v2')
            if not venv.exists():
                subprocess.run(['uv', 'venv', '-q', '--python', '3.11', str(venv)], check=True)
            subprocess.run(['uv', 'pip', 'install', '-q', '--python', str(venv / 'bin/python'),
                            '--index-url', 'https://download.pytorch.org/whl/cu124',
                            'torch==2.6.0', 'torchvision==0.21.0'], check=True)
            subprocess.run(['uv', 'pip', 'install', '-q', '--python', str(venv / 'bin/python'),
                            'inference-gpu==1.8.3', 'onnxruntime-gpu[cuda,cudnn]==1.21.1',
                            'torch==2.6.0', 'torchvision==0.21.0'], check=True)
            freeze = subprocess.run(['uv', 'pip', 'freeze', '--python', str(venv / 'bin/python')],
                                    check=True, capture_output=True, text=True)
            (OUT / 'rf_environment.txt').write_text(freeze.stdout)
            args = [str(venv / 'bin/python'), str(RF_SCRIPT)]
            for clip, path in RF_MISSING_CLIPS.items():
                args += [str(path), str(OUT / f'rf_raw_{clip}.json')]
            completed = subprocess.run(args, env=dict(os.environ, ROBOFLOW_API_KEY=ROBOFLOW_API_KEY,
                RF_MODEL_ID=RF_MODEL_ID, RF_SWEEP=str(RF_SWEEP).lower(),
                RF_VARIANTS=','.join(RF_SELECTED_VARIANTS or ()), ACTIVE_LEARNING_ENABLED='false'),
                capture_output=True, text=True)
            print((completed.stdout + completed.stderr).replace(ROBOFLOW_API_KEY, '[REDACTED]'))
            if completed.returncode:
                raise RuntimeError('RF-DETR inference failed; see redacted output above')
        raw = json.loads((OUT / f'rf_raw_{next(iter(CLIPS))}.json').read_text())
        for label in RF_LABELS:
            MIN_CONF[label] = raw['variants'][label]['config']['tracker_min_conf']
            run_model(label, rf_detector(label, label))
            if COMPARE_KALMAN:
                output_label = label+'-Kalman'
                MIN_CONF[output_label] = MIN_CONF[label]
                def kalman_tracking(detections, width, height, fps, confidence):
                    return track_temporal(detections, width, height, fps,
                                          TemporalConfig(min_confidence=confidence, **KALMAN_SETTINGS))
                run_model(output_label, rf_detector(label, output_label, temporal=True), tracking=kalman_tracking)
    except Exception as error:
        message = str(error).replace(ROBOFLOW_API_KEY, '[REDACTED]') if ROBOFLOW_API_KEY else str(error)
        for label in RF_LABELS:
            RESULTS.setdefault(label, {clip: dict(error=message) for clip in CLIPS})
        (OUT / 'rf_setup_error.json').write_text(json.dumps(dict(error=message)))
        print('RF-DETR setup failed:', message)
else:
    print('RF-DETR skipped (no key or cached detections).')

## 3. SAM 3 video, text prompt "basketball"
Meta's model finds every object matching the text and follows each one through the video
using memory (it reads the whole clip first, so this is offline processing).

In [ ]:
SAM3_PROMPT = 'basketball'

if RUN_OPTIONAL_MODELS and not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass('Hugging Face read token (hidden, session only; Enter to skip): ').strip()

if RUN_OPTIONAL_MODELS and HF_TOKEN:
    try:
        import inspect
        from transformers import Sam3VideoModel, Sam3VideoProcessor
        dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
        sam3 = Sam3VideoModel.from_pretrained('facebook/sam3', token=HF_TOKEN).to('cuda', dtype=dtype).eval()
        sam3_processor = Sam3VideoProcessor.from_pretrained('facebook/sam3', token=HF_TOKEN)

        def to_numpy(value):
            return value.detach().float().cpu().numpy() if hasattr(value, 'detach') else np.asarray(value, float)

        def sam3_detect(clip, frames):
            options = dict(video=np.stack([cv2.cvtColor(f, cv2.COLOR_BGR2RGB) for f in frames]),
                           inference_device='cuda', processing_device='cpu', video_storage_device='cpu')
            if 'dtype' in inspect.signature(sam3_processor.init_video_session).parameters:
                options['dtype'] = dtype
            session = sam3_processor.init_video_session(**options)
            sam3_processor.add_text_prompt(inference_session=session, text=SAM3_PROMPT)
            found = [[] for _ in frames]
            torch.cuda.synchronize(); t0 = time.time()
            with torch.inference_mode():
                for output in sam3.propagate_in_video_iterator(inference_session=session):
                    r = sam3_processor.postprocess_outputs(session, output)
                    ids = to_numpy(r['object_ids']).reshape(-1)
                    for oid, conf, (x0, y0, x1, y1) in zip(ids, to_numpy(r['scores']).reshape(-1),
                                                          to_numpy(r['boxes']).reshape(-1, 4)):
                        found[output.frame_idx].append(dict(x=float((x0 + x1) / 2), y=float((y0 + y1) / 2),
                                                            r=float(max(x1 - x0, y1 - y0) / 2),
                                                            conf=float(conf), sam3_id=int(oid)))
            torch.cuda.synchronize()
            return found, time.time() - t0

        run_model('SAM3', sam3_detect)
        del sam3; torch.cuda.empty_cache()
    except Exception as error:
        message = str(error).replace(HF_TOKEN, '[REDACTED]')
        for clip in CLIPS:
            RESULTS.setdefault('SAM3', {}).setdefault(clip, dict(error=message))
        (OUT / 'sam3_setup_error.json').write_text(json.dumps(dict(error=message)))
        print('SAM 3 unavailable; continuing to summary:', message)
    finally:
        if 'sam3' in globals():
            del sam3
        torch.cuda.empty_cache()
else:
    print('SAM3 skipped (optional models disabled or token missing).')

## Side-by-side comparison and summary
Every successful variant appears in synchronized comparison videos. Each page has
the original and up to three variants so the ball stays visible. Individual full-size
videos above remain available. All rows (including failures) are saved in summary.csv.

In [ ]:
import pandas as pd

for clip, path in CLIPS.items():
    frames, fps = read_video(path)
    scale = 640 / frames[0].shape[1]
    labels = [label for label in RESULTS if 'tracked' in RESULTS[label].get(clip, {})
              and (SHOW_BASELINE or not COMPARE_KALMAN or label.endswith('-Kalman'))]
    for page, start in enumerate(range(0, len(labels), 3), 1):
        selected = labels[start:start+3]
        def comparison_frames():
            for i, f in enumerate(frames):
                panels = [cv2.resize(f, None, fx=scale, fy=scale)]
                cv2.putText(panels[0], 'Original', (8, 24), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255,255,255), 2)
                for label in selected:
                    r = RESULTS[label][clip]
                    panels.append(draw(f, r['detections'], r['tracked'], i, label, scale))
                while len(panels) < 4:
                    panels.append(np.zeros_like(panels[0]))
                yield np.vstack([np.hstack(panels[:2]), np.hstack(panels[2:])])
        video = OUT / f'comparison_{clip}_page{page}.mp4'
        # Stream composition instead of retaining a full 1280x720 grid in memory.
        write_video(video, comparison_frames(), fps, frame_shape=(720, 1280))
        print(clip, 'comparison', page, selected)
        show(video, width=1100)

rows = [dict(model=label, clip=clip, **r.get('stats', {'error': r.get('error')}))
        for label, clips in RESULTS.items() for clip, r in clips.items()]
summary = pd.DataFrame(rows)
summary.to_csv(OUT / 'summary.csv', index=False)
display(summary)
display(Markdown('''
- **raw_found**: of the hand-marked check frames (8 in clip 1, 11 in clip 2), how many had *some* detection on the ball.
- **tracked_correct / tracked_wrong_place**: the tracked ball was on the real ball / somewhere else.
- **check_off_target_candidates**: candidates outside the draft reference radius on checked frames (duplicates on-target are not penalized). Not a complete precision metric.
- Interpolated/predicted positions are inferred, excluded from observed-hit counts. Tracker time is separate from detector time, which is historical when cached.
- The 19 draft checkpoints are development feedback, not held-out ground truth. Coverage and smooth overlays are not accuracy; evaluate separately before selecting a winner.
'''))
print('Everything saved in', OUT)